In [1]:
import torch
import yaml
import types
import json
import numpy as np
import sys
dir = '/home/jjwhit/rcGAN/'
sys.path.append(dir)

from data.lightning.MassMappingDataModule import MMDataModule
from utils.parse_args import create_arg_parser
from pytorch_lightning import seed_everything
from models.lightning.mmGAN import mmGAN
from mass_map_utils.scripts.transforms import tensor_to_complex_np
from scipy import ndimage

In [2]:
exp_name = "mmgan_training_crps"

In [3]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

In [4]:
def load_object(dct):
    return types.SimpleNamespace(**dct)

In [5]:
torch.set_float32_matmul_precision('medium')

with open(dir+'/configs/mass_map.yml', 'r') as f:
    cfg = yaml.load(f, Loader=yaml.FullLoader)
    cfg = json.loads(json.dumps(cfg), object_hook=load_object)

dm = MMDataModule(cfg)
dm.setup()
test_loader = dm.test_dataloader()

with torch.no_grad():
    mmGAN_model = mmGAN.load_from_checkpoint(
        checkpoint_path=(cfg.checkpoint_dir + exp_name + '/checkpoint_best.ckpt'))

    mmGAN_model.cuda()

    mmGAN_model.eval()

In [7]:
data = next(iter(test_loader))

y, x, mean, std = data

print(y.shape, x.shape)

/home/jjwhit/rcGAN/data/lightning/MassMappingDataModule.py:100: RuntimeWarning: divide by zero encountered in true_divide
  F_kappa = F_gamma / D
/home/jjwhit/rcGAN/data/lightning/MassMappingDataModule.py:100: RuntimeWarning: divide by zero encountered in true_divide
  F_kappa = F_gamma / D
/home/jjwhit/rcGAN/data/lightning/MassMappingDataModule.py:100: RuntimeWarning: divide by zero encountered in true_divide
  F_kappa = F_gamma / D
/home/jjwhit/rcGAN/data/lightning/MassMappingDataModule.py:100: RuntimeWarning: divide by zero encountered in true_divide
  F_kappa = F_gamma / D


torch.Size([9, 4, 300, 300]) torch.Size([9, 1, 300, 300])


In [8]:
# Reduce batch size for debugging
y = y[:1]
x = x[:1]
mean = mean[:1]
std = std[:1]

print("y:", y.shape)
print("x:", x.shape)

y: torch.Size([1, 4, 300, 300])
x: torch.Size([1, 1, 300, 300])


In [10]:
with torch.no_grad():
    zfr = mmGAN_model.reformat(y)

print("zfr shape:", zfr.shape)
print("zfr dtype:", zfr.dtype)

zfr shape: torch.Size([1, 300, 300, 4])
zfr dtype: torch.float32


In [11]:
j = 0

kappa_mean = cfg.kappa_mean
kappa_std = cfg.kappa_std

In [12]:
scaled = zfr[j] * kappa_std + kappa_mean

print("scaled shape:", scaled.shape)
print("scaled dtype:", scaled.dtype)

scaled shape: torch.Size([300, 300, 4])
scaled dtype: torch.float32


In [16]:
data = next(iter(test_loader))
y, x, _, _ = data

print("y channels:", y.shape[1])

# inspect channels
for c in range(y.shape[1]):
    print(f"channel {c} stats:", y[0, c].mean().item(), y[0, c].std().item())

/home/jjwhit/rcGAN/data/lightning/MassMappingDataModule.py:100: RuntimeWarning: divide by zero encountered in true_divide
  F_kappa = F_gamma / D
/home/jjwhit/rcGAN/data/lightning/MassMappingDataModule.py:100: RuntimeWarning: divide by zero encountered in true_divide
  F_kappa = F_gamma / D
/home/jjwhit/rcGAN/data/lightning/MassMappingDataModule.py:100: RuntimeWarning: divide by zero encountered in true_divide
  F_kappa = F_gamma / D
/home/jjwhit/rcGAN/data/lightning/MassMappingDataModule.py:100: RuntimeWarning: divide by zero encountered in true_divide
  F_kappa = F_gamma / D


y channels: 4
channel 0 stats: 0.002163211116567254 0.6182962656021118
channel 1 stats: -0.00035808299435302615 0.6141196489334106
channel 2 stats: 0.004823682829737663 0.9848974347114563
channel 3 stats: 0.0004818450834136456 0.9753644466400146


RuntimeError: Tensor must have a last dimension of size 2